In [1]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml.metrics import mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score, accuracy_score, confusion_matrix, kl_divergence
from cuml.metrics import log_loss, roc_auc_score, nan_euclidean_distances, pairwise_distances, sparse_pairwise_distances
from cuml.model_selection import train_test_split, KFold
from cuml.linear_model import LogisticRegression, MBSGDClassifier
from cuml.multiclass import OneVsRestClassifier, OneVsOneClassifier 

In [2]:
df = cudf.read_csv('heart_disease_health_indicators_BRFSS2015.csv')
df

,HeartDiseaseorAttack,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,Diabetes,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253675,0.0,1.0,1.0,1.0,45.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,5.0,0.0,1.0,5.0,6.0,7.0
253676,0.0,1.0,1.0,1.0,18.0,0.0,0.0,2.0,0.0,0.0,...,1.0,0.0,4.0,0.0,0.0,1.0,0.0,11.0,2.0,4.0
253677,0.0,0.0,0.0,1.0,28.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,2.0,5.0,2.0
253678,0.0,1.0,0.0,1.0,23.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,7.0,5.0,1.0


In [3]:
from ipynb.fs.full.Normalization_MultiClass_to_import import *

Execution time: 0.7597434520721436 seconds


In [4]:
from ipynb.fs.full.GeneratedDataMultiClass_to_import import *

Execution time: 0.05704140663146973 seconds


In [5]:
torch.cuda.empty_cache()
gc.collect()

0

In [6]:
class MultiClass(object):
    def __init__(self, dataset, generated_data_multiclass_global_1):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.X = cudf.DataFrame(self.dataset.copy().drop(['GenHlth'], axis = 1))
        self.y = cudf.DataFrame(self.dataset['GenHlth'].copy())
        self.generated_data_multiclass_global_1 = generated_data_multiclass_global_1.copy().reset_index(drop = True)
        self.dataset_numpy_array = self.dataset.copy().to_numpy()
        self.X_cupy_array = self.X.to_cupy()
        self.y_cupy_array = self.y.to_cupy()
        self.gen_data_cupy_array = self.generated_data_multiclass_global_1.to_cupy()


    def train_test_split(self):
        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        X_train, X_test, y_train, y_test = train_test_split(self.X_cupy_array, self.y_cupy_array, test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def OneVsOneClassifier(self):
        global model_ovoc_global 
        global ovoc_global
        global ovoc_predict_test_global
        global ovoc_predict_global_1
        global mean_squared_error_global_ovoc
        global mean_squared_log_error_global_ovoc
        global median_absolute_error_global_ovoc
        global r2_score_global_ovoc
        global accuracy_score_global_ovoc
        global kl_divergence_global_ovoc
        global log_loss_global_ovoc
        global roc_auc_score_global_ovoc
        global nan_euclidean_distances_global_ovoc
        global pairwise_distances_global_ovoc
        global sparse_pairwise_distances_global_ovoc

        model = OneVsOneClassifier(MBSGDClassifier(loss='hinge', penalty='l2', alpha=0.0001, l1_ratio=0.15, 
                                                       fit_intercept=True, epochs=1000, tol=0.001, 
                                                       shuffle=True, learning_rate='constant', eta0=0.001, 
                                                       power_t=0.5, batch_size=32, n_iter_no_change=5, verbose=False, 
                                                       output_type=None))
        model_ovoc_global = model
            
        ovoc = model_ovoc_global.fit(X_train_global.copy(), y_train_global.copy().ravel())
        ovoc_global = ovoc
        preds_test = model_ovoc_global.predict(X_test_global.copy())
        ovoc_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['GenHlth']).reset_index(drop = True)
        
        mean_squared_error_global_ovoc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_ovoc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_ovoc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_ovoc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_ovoc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_ovoc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_ovoc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_ovoc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_ovoc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_ovoc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_ovoc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model_ovoc_global.predict(self.gen_data_cupy_array)
        ovoc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['GenHlth']).reset_index(drop = True) 

    def OneVsRestClassifier(self):
        global model_ovrc_global
        global ovrc_global
        global ovrc_predict_test_global
        global ovrc_predict_global_1
        global mean_squared_error_global_ovrc
        global mean_squared_log_error_global_ovrc
        global median_absolute_error_global_ovrc
        global r2_score_global_ovrc
        global accuracy_score_global_ovrc
        global kl_divergence_global_ovrc
        global log_loss_global_ovrc
        global roc_auc_score_global_ovrc
        global nan_euclidean_distances_global_ovrc
        global pairwise_distances_global_ovrc
        global sparse_pairwise_distances_global_ovrc

        model = OneVsRestClassifier(MBSGDClassifier(loss='hinge', penalty='l2', alpha=0.0001, l1_ratio=0.15, 
                                                       fit_intercept=True, epochs=1000, tol=0.001, 
                                                       shuffle=True, learning_rate='constant', eta0=0.001, 
                                                       power_t=0.5, batch_size=32, n_iter_no_change=5, verbose=False, 
                                                       output_type=None))
        model_ovrc_global = model
            
        ovrc = model_ovrc_global.fit(X_train_global.copy(), y_train_global.copy().ravel())
        ovrc_global = ovrc
        preds_test = model_ovrc_global.predict(X_test_global.copy())
        ovrc_predict_test_global = cudf.DataFrame(preds_test.copy(), columns = ['GenHlth']).reset_index(drop = True)
        
        mean_squared_error_global_ovrc = mean_squared_error(y_test_global.copy(), preds_test)
        #mean_squared_log_error_global_ovrc = mean_squared_log_error(y_test_global.copy(), preds_test)
        median_absolute_error_global_ovrc = median_absolute_error(y_test_global.copy(), preds_test)
        r2_score_global_ovrc = r2_score(y_test_global.copy(), preds_test)
        accuracy_score_global_ovrc = accuracy_score(y_test_global.copy(), preds_test)
        kl_divergence_global_ovrc = kl_divergence(y_test_global.copy(), preds_test)
        #log_loss_global_ovrc = log_loss(y_test_global.copy(), preds_test)
        roc_auc_score_global_ovrc = roc_auc_score(y_test_global.copy(), preds_test)
        #nan_euclidean_distances_global_ovrc = nan_euclidean_distances(y_test_global.copy(), preds_test)
        #pairwise_distances_global_ovrc = pairwise_distances(y_test_global.copy(), preds_test)
        #sparse_pairwise_distances_global_ovrc = sparse_pairwise_distances(y_test_global.copy(), preds_test)
     
        preds_1 = model_ovrc_global.predict(self.gen_data_cupy_array)
        ovrc_predict_global_1 = cudf.DataFrame(preds_1.copy(), columns = ['GenHlth']).reset_index(drop = True)

    def main(self):
        st = time.time()
        self.train_test_split()
        self.OneVsOneClassifier()
        self.OneVsRestClassifier()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [7]:
class Metrics(object):
    def OneVsOneClassifier(self):
        print('OneVsOneClassifier: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_ovoc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_ovoc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_ovoc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_ovoc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_ovoc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_ovoc)
        print('\n')

    def OneVsRestClassifier(self):
        print('OneVsRestClassifier: ')
        print('Mean squared error: ')
        print(mean_squared_error_global_ovrc)
        print('\n')
        print('Median Absolute Error: ')
        print(median_absolute_error_global_ovrc)
        print('\n')
        print('R2 Score: ')
        print(r2_score_global_ovrc)
        print('\n')
        print('Accuracy Score: ')
        print(accuracy_score_global_ovrc)
        print('\n')
        print('Kl Divergence: ')
        print(kl_divergence_global_ovrc)
        print('\n')
        print('ROC AUC Score: ')
        print(roc_auc_score_global_ovrc)
        print('\n')

    def main(self):
        self.OneVsOneClassifier()
        self.OneVsRestClassifier()

In [8]:
torch.cuda.empty_cache()
gc.collect()

0

In [9]:
multiclass_MBSGDClassifier = MultiClass(df, generated_data_multiclass_global)
multiclass_MBSGDClassifier.main()
multiclass_MBSGDClassifier_metrics = Metrics()
print("Metrics for not normalized data is ready. Run 'multiclass_MBSGDClassifier_metrics.main()'!")
#multiclass_MBSGDClassifier_metrics.main()

Execution time: 2614.53155708313 seconds
Metrics for not normalized data is ready. Run 'multiclass_MBSGDClassifier_metrics.main()'!


In [10]:
torch.cuda.empty_cache()
gc.collect()

73

In [11]:
multiclass_maxAbsScaler_MBSGDClassifier = MultiClass(maxAbsScaler_merged_multiclass_global, 
                                                     generated_data_multiclass_global)
multiclass_maxAbsScaler_MBSGDClassifier.main()
multiclass_maxAbsScaler_MBSGDClassifier_metrics = Metrics()
print("Metrics for maxAbsScaler normalized data is ready. Run 'multiclass_maxAbsScaler_MBSGDClassifier_metrics.main()'!")
#multiclass_maxAbsScaler_MBSGDClassifier_metrics.main()

Execution time: 2745.8796553611755 seconds
Metrics for maxAbsScaler normalized data is ready. Run 'multiclass_maxAbsScaler_MBSGDClassifier_metrics.main()'!


In [12]:
torch.cuda.empty_cache()
gc.collect()

74

In [13]:
multiclass_MinMaxScaler_MBSGDClassifier = MultiClass(MinMaxScaler_merged_multiclass_global, 
                                                     generated_data_multiclass_global)
multiclass_MinMaxScaler_MBSGDClassifier.main()
multiclass_MinMaxScaler_MBSGDClassifier_metrics = Metrics()
print("Metrics for MinMaxScaler normalized data is ready. Run 'multiclass_MinMaxScaler_MBSGDClassifier_metrics.main()'!")
#multiclass_MinMaxScaler_MBSGDClassifier_metrics.main()

Execution time: 2735.4721319675446 seconds
Metrics for MinMaxScaler normalized data is ready. Run 'multiclass_MinMaxScaler_MBSGDClassifier_metrics.main()'!


In [14]:
torch.cuda.empty_cache()
gc.collect()

74

In [15]:
multiclass_Normalizer_MBSGDClassifier = MultiClass(normalizer_merged_multiclass_global, 
                                           generated_data_multiclass_global)
multiclass_Normalizer_MBSGDClassifier.main()
multiclass_Normalizer_MBSGDClassifier_metrics = Metrics()
print("Metrics for Normalizer normalized data is ready. Run 'multiclass_Normalizer_MBSGDClassifier_metrics.main()'!")
#multiclass_Normalizer_MBSGDClassifier_metrics.main()

Execution time: 2708.254734992981 seconds
Metrics for Normalizer normalized data is ready. Run 'multiclass_Normalizer_MBSGDClassifier_metrics.main()'!


In [16]:
torch.cuda.empty_cache()
gc.collect()

74

In [20]:
multiclass_RobustScaler_MBSGDClassifier  = MultiClass(robust_scaler_merged_multiclass_global, 
                                           generated_data_multiclass_global)
multiclass_RobustScaler_MBSGDClassifier.main()
multiclass_RobustScaler_MBSGDClassifier_metrics = Metrics()
print("Metrics for RobustScaler normalized data is ready. Run 'multiclass_RobustScaler_MBSGDClassifier_metrics.main()'!")
#multiclass_RobustScaler_MBSGDClassifier_metrics.main()

Execution time: 2726.5643537044525 seconds
Metrics for RobustScaler normalized data is ready. Run 'multiclass_RobustScaler_MBSGDClassifier_metrics.main()'!


In [21]:
torch.cuda.empty_cache()
gc.collect()

74

In [22]:
multiclass_StandardScaler_MBSGDClassifier  = MultiClass(standard_scaler_merged_multiclass_global, 
                                                  generated_data_multiclass_global)
multiclass_StandardScaler_MBSGDClassifier.main()
multiclass_StandardScaler_MBSGDClassifier_metrics = Metrics()
print("Metrics for StandardScaler normalized data is ready. Run 'multiclass_StandardScaler_MBSGDClassifier_metrics.main()'!")
#multiclass_StandardScaler_MBSGDClassifier _metrics.main()

Execution time: 2730.039913415909 seconds
Metrics for StandardScaler normalized data is ready. Run 'multiclass_StandardScaler_MBSGDClassifier_metrics.main()'!


In [23]:
torch.cuda.empty_cache()
gc.collect()

74

In [24]:
multiclass_Binarizer_MBSGDClassifier = MultiClass(binarizer_merged_multiclass_global, 
                                                  generated_data_multiclass_global)
multiclass_Binarizer_MBSGDClassifier.main()
multiclass_Binarizer_MBSGDClassifier_metrics = Metrics()
print("Metrics for Binarizer normalized data is ready. Run 'multiclass_Binarizer_MBSGDClassifier_metrics.main()'!")
#multiclass_Binarizer_MBSGDClassifier_metrics.main()

Execution time: 2733.9279692173004 seconds
Metrics for Binarizer normalized data is ready. Run 'multiclass_Binarizer_MBSGDClassifier_metrics.main()'!


In [25]:
torch.cuda.empty_cache()
gc.collect()

74

In [27]:
multiclass_FunctionTransformerMBSGDClassifier = MultiClass(function_transformer_merged_multiclass_global, 
                                                  generated_data_multiclass_global)
multiclass_FunctionTransformerMBSGDClassifier.main()
multiclass_FunctionTransformerMBSGDClassifier_metrics = Metrics()
print("Metrics for FunctionTransformer normalized data is ready. Run 'multiclass_FunctionTransformerMBSGDClassifier_metrics.main()'!")
#multiclass_FunctionTransformerMBSGDClassifier_metrics.main()

Execution time: 2705.2049553394318 seconds
Metrics for FunctionTransformer normalized data is ready. Run 'multiclass_FunctionTransformerMBSGDClassifier_metrics.main()'!


In [28]:
torch.cuda.empty_cache()
gc.collect()

557

In [29]:
multiclass_KBinsDiscretizer_MBSGDClassifier = MultiClass(KBinsDiscretizer_merged__multiclass_global, 
                                                  generated_data_multiclass_global)
multiclass_KBinsDiscretizer_MBSGDClassifier.main()
multiclass_KBinsDiscretizer_MBSGDClassifier_metrics = Metrics()
print("Metrics for KBinsDiscretizer normalized data is ready. Run 'multiclass_KBinsDiscretizer_MBSGDClassifier_metrics.main()'!")
#multiclass_KBinsDiscretizer_MBSGDClassifier_metrics.main()

Execution time: 2709.9341382980347 seconds
Metrics for KBinsDiscretizer normalized data is ready. Run 'multiclass_KBinsDiscretizer_MBSGDClassifier_metrics.main()'!


In [30]:
torch.cuda.empty_cache()
gc.collect()

74

In [31]:
multiclass_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [32]:
multiclass_maxAbsScaler_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [33]:
multiclass_MinMaxScaler_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [34]:
multiclass_Normalizer_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [35]:
multiclass_RobustScaler_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [36]:
multiclass_StandardScaler_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [37]:
multiclass_Binarizer_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [38]:
multiclass_FunctionTransformerMBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083




In [39]:
multiclass_KBinsDiscretizer_MBSGDClassifier_metrics.main()

OneVsOneClassifier: 
Mean squared error: 
1.3708215074109114


Median Absolute Error: 
1.0


R2 Score: 
-0.19861535631405003


Accuracy Score: 
0.36443550930305896


Kl Divergence: 
13336.638782029999


ROC AUC Score: 
0.4530056118965149


OneVsRestClassifier: 
Mean squared error: 
1.2049235257016715


Median Absolute Error: 
1.0


R2 Score: 
-0.05355790909484992


Accuracy Score: 
0.37961210974456006


Kl Divergence: 
6534.068874391075


ROC AUC Score: 
0.4616033732891083


